# PoC de reunião com LangChain 

## 0. Instalando dependências

In [1]:
%pip install -q python-dotenv pydantic langchain-core langchain-google-genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports

In [2]:
import os

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

## 2. Chave de API

In [3]:
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

## 3. Leitura da transcrição

O texto da reunião é obtido de `meeting-text.txt` e será enviado ao modelo como contexto.

In [4]:
with open("meeting-text.txt", "r", encoding="utf-8") as meeting_text_file:
    text = meeting_text_file.read()

print("Texto da reunião carregado com sucesso.")

Texto da reunião carregado com sucesso.


## 4. Formato esperado da resposta

O modelo deve retornar um objetivo principal e uma lista de tarefas. O Pydantic valida essa estrutura depois que a resposta é recebida.

In [5]:
class MeetingSummary(BaseModel):
    tasks: list[str] = Field(..., description="Lista de tarefas extraídas da reunião.")

    objective: str = Field(..., description="Objetivo principal extraído da reunião.")


parser = PydanticOutputParser(pydantic_object=MeetingSummary)

print("PydanticOutputParser criado com sucesso")

PydanticOutputParser criado com sucesso


## 5. Modelo e prompt

Configuramos o Gemini com temperatura zero para respostas mais consistentes. O parser inclui no prompt as instruções necessárias para o retorno seguir a estrutura definida acima.

In [6]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0,
)

prompt = PromptTemplate(
    input_variables=["meeting_text"],
    template="""
Você é um assistente de inteligência artificial especializado
em extrair informações de reuniões.

Analise a reunião abaixo e extraia:
o objetivo principal;
as tarefas definidas.

Não invente informações que não estejam presentes na reunião.

Reunião:
{meeting_text}

{format_instructions}
""",
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    },
)

print("PromptTemplate criado com sucesso.")

PromptTemplate criado com sucesso.


## 6. Montagem e envio do prompt

Nesta etapa inserimos a transcrição no template e enviamos o prompt completo ao Gemini. A execução desta célula requer uma chave de API válida.

In [7]:
prompt_final = prompt.format(meeting_text=text)

llm_response = llm.invoke(prompt_final)

print("Resposta do modelo recebida com sucesso.")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Resposta do modelo recebida com sucesso.


## 7. Validação e Resposta final

Primeiro exibimos a resposta bruta para facilitar a depuração. Em seguida, o parser converte e valida o conteúdo no modelo `MeetingSummary`, e mostramos o resultado como dicionário.

In [8]:
# RESPOSTA BRUTA
print(llm_response.content)

```json
{
  "objective": "Fechar o planejamento da próxima sprint, revisar o que ficou pendente da anterior e dividir as tasks.",
  "tasks": [
    "Maria: Fechar o Pull Request (PR) de autenticação (refresh token).",
    "Maria: Implementar o tratamento de sessão expirada.",
    "Maria: Atualizar a documentação dos endpoints (critérios de aceite no ticket e comportamento final na documentação).",
    "Pedro: Fechar o escopo do dashboard (incluindo comportamento dos filtros, skeleton loading, paginação de 20 registros configurável).",
    "Pedro: Mandar o escopo do dashboard para revisão até amanhã.",
    "Pedro: Criar as tasks do dashboard no board, incluindo critérios de aceite no ticket principal.",
    "Pedro: Enviar os nomes finais das métricas para Bia até sexta de manhã.",
    "Bia: Desenvolver os componentes iniciais do dashboard (cards, tabela, estado vazio) usando o design system.",
    "Bia: Criar um componente de filtro reutilizável.",
    "Bia: Garantir que os estados de er

In [9]:
# Print com resposta final do modelo, já parseada pelo PydanticOutputParser

parsed_output = parser.parse(llm_response.content)
print(parsed_output.model_dump())

{'tasks': ['Maria: Fechar o Pull Request (PR) de autenticação (refresh token).', 'Maria: Implementar o tratamento de sessão expirada.', 'Maria: Atualizar a documentação dos endpoints (critérios de aceite no ticket e comportamento final na documentação).', 'Pedro: Fechar o escopo do dashboard (incluindo comportamento dos filtros, skeleton loading, paginação de 20 registros configurável).', 'Pedro: Mandar o escopo do dashboard para revisão até amanhã.', 'Pedro: Criar as tasks do dashboard no board, incluindo critérios de aceite no ticket principal.', 'Pedro: Enviar os nomes finais das métricas para Bia até sexta de manhã.', 'Bia: Desenvolver os componentes iniciais do dashboard (cards, tabela, estado vazio) usando o design system.', 'Bia: Criar um componente de filtro reutilizável.', 'Bia: Garantir que os estados de erro e loading estejam bem definidos nos componentes.', 'Bia: Entregar os componentes iniciais do dashboard e o filtro reutilizável até sexta.', 'João: Verificar o deploy (te

## 8. Simulação de limitação de tamanho com TextSplitters

Transcrições podem ultrapassar a janela de contexto de um modelo. Para demonstrar como o LangChain consegue dividir um texto grande em partes menores, utilizamos o `CharacterTextSplitter`.

In [10]:
%pip install -q langchain-text-splitters
%pip install -q tiktoken

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(separator="\n", chunk_size=500, chunk_overlap=50, length_function=len)

chunks = text_splitter.split_text(text)

print(f"Tamanho do texto original: {len(text)} caracteres")
print(f"Quantidade de partes geradas: {len(chunks)}")

for indice, chunk in enumerate(chunks, start=1):
    print(f"\n--- Parte {indice} | {len(chunk)} caracteres ---")
    print(chunk)

Created a chunk of size 546, which is longer than the specified 500


Tamanho do texto original: 16377 caracteres
Quantidade de partes geradas: 42

--- Parte 1 | 496 caracteres ---
[00:00] Oi, gente. Estão me ouvindo? Acho que meu microfone tava meio estranho aqui.
[00:05] Tá ouvindo sim. Bom dia, pessoal.
[00:09] Bom diaa. Eu entrei agora, foi mal, a internet deu uma travada.
[00:14] Relaxa. Vamos esperar só mais um minutinho pelo Lucas e pela Carla. A Maria já falou que vai entrar pelo celular porque tá saindo de uma aula.
[00:32] Eu tô aqui também, mas sem câmera porque tô comendo. Não me julguem.
[00:37] Justo, daily cedo é isso. Ah, Lucas entrou. Falta só a Carla.

--- Parte 2 | 412 caracteres ---
[00:52] Opa, desculpa. Eu achei que era nove e dez. Foi mal mesmo.
[00:58] Não, era nove, mas tá tudo certo. A Carla mandou mensagem, vai entrar em cinco minutos. A gente já começa. A ideia hoje é fechar o planejamento da próxima sprint, revisar o que ficou pendente da anterior e dividir as tasks. Sem apresentação, sem enrolar muito, porque todo mundo tem 

### Resultado esperado

A saída mostra quantas partes foram criadas e o tamanho de cada uma. Assim, em vez de enviar uma transcrição muito grande de uma única vez ao modelo, é possível processar os blocos separadamente, reduzindo o risco de ultrapassar a janela de contexto.

## 9. Divisão por tokens com TokenTextSplitter

Enquanto o `CharacterTextSplitter` mede o tamanho dos blocos em caracteres, o `TokenTextSplitter` trabalha com **tokens**, que são a unidade usada pelos modelos de linguagem para medir sua janela de contexto.

Neste exemplo, a mesma transcrição é dividida em blocos de até 500 tokens, com uma sobreposição de 50 tokens entre blocos consecutivos.

In [12]:
from langchain_text_splitters import TokenTextSplitter

token_splitter = TokenTextSplitter(chunk_size=500, chunk_overlap=50)

token_chunks = token_splitter.split_text(text)

print(f"Quantidade de partes geradas: {len(token_chunks)}")

for indice, chunk in enumerate(token_chunks, start=1):
    print(f"\n--- Parte {indice} ---")
    print(chunk)

Quantidade de partes geradas: 15

--- Parte 1 ---
[00:00] Oi, gente. Estão me ouvindo? Acho que meu microfone tava meio estranho aqui.

[00:05] Tá ouvindo sim. Bom dia, pessoal.

[00:09] Bom diaa. Eu entrei agora, foi mal, a internet deu uma travada.

[00:14] Relaxa. Vamos esperar só mais um minutinho pelo Lucas e pela Carla. A Maria já falou que vai entrar pelo celular porque tá saindo de uma aula.

[00:32] Eu tô aqui também, mas sem câmera porque tô comendo. Não me julguem.

[00:37] Justo, daily cedo é isso. Ah, Lucas entrou. Falta só a Carla.

[00:52] Opa, desculpa. Eu achei que era nove e dez. Foi mal mesmo.

[00:58] Não, era nove, mas tá tudo certo. A Carla mandou mensagem, vai entrar em cinco minutos. A gente já começa. A ideia hoje é fechar o planejamento da próxima sprint, revisar o que ficou pendente da anterior e dividir as tasks. Sem apresentação, sem enrolar muito, porque todo mundo tem coisa depois.

[01:19] Graças a Deus, reunião objetiva.

[01:22] É o plano. Então, só pr

### Comparação

- `CharacterTextSplitter`: limita os blocos com base na quantidade de caracteres.
- `TokenTextSplitter`: limita os blocos com base na quantidade de tokens.

Para trabalhar diretamente com limites de contexto de modelos de linguagem, o `TokenTextSplitter` representa melhor o problema real, pois as janelas de contexto dos modelos são definidas em tokens.